# 300. Longest Increasing Subsequence

## Topic Alignment
- LIS is fundamental in version control systems for finding longest chains of dependencies, task scheduling with precedence constraints, and analyzing monotonic trends in time series data for ML feature engineering.

## Metadata Summary
- Source: https://leetcode.com/problems/longest-increasing-subsequence/
- Tags: Dynamic Programming, Binary Search, Array
- Difficulty: Medium
- Priority: High

## Problem Statement
Given an integer array `nums`, return the length of the longest **strictly increasing subsequence**.

A **subsequence** is an array that can be derived from another array by deleting some or no elements without changing the order of the remaining elements.

## Progressive Hints
- Hint 1: For each position i, compute the longest increasing subsequence ending at i.
- Hint 2: To find LIS ending at i, check all previous positions j < i where nums[j] < nums[i].
- Hint 3: The O(n^2) DP solution uses dp[i] = max(dp[j]) + 1 for all valid j.
- Hint 4: For O(n log n), maintain an array that tracks the smallest tail element for each length.
- Hint 5: Use binary search to find the position to update in the tail array.

## Solution Overview
This classic problem has two main approaches:

**1. Dynamic Programming O(n^2)**
- `dp[i]` = length of LIS ending at index i
- For each i, check all j < i: if nums[j] < nums[i], then dp[i] = max(dp[i], dp[j] + 1)
- Answer is max(dp)

**2. Binary Search O(n log n)**
- Maintain a `tails` array where `tails[i]` is the smallest tail element of all increasing subsequences of length i+1
- For each number, binary search to find its position in tails
- Replace or append to maintain the invariant
- Length of tails is the answer

The binary search approach is more efficient and demonstrates an elegant optimization technique.

## Detailed Explanation

### Approach 1: Dynamic Programming O(n^2)

**Core Idea**: Build up the solution by considering each element as a potential end of an increasing subsequence.

1. **State Definition**: `dp[i]` = length of the longest increasing subsequence that ends with `nums[i]`

2. **Recurrence Relation**:
   ```
   dp[i] = max(dp[j] + 1) for all j < i where nums[j] < nums[i]
   ```
   If no such j exists, `dp[i] = 1` (subsequence of just nums[i])

3. **Base Case**: `dp[i] = 1` for all i (each element forms a subsequence of length 1)

4. **Algorithm**:
   - Initialize dp array with all 1s
   - For i from 0 to n-1:
     - For j from 0 to i-1:
       - If nums[j] < nums[i]: dp[i] = max(dp[i], dp[j] + 1)
   - Return max(dp)

**Example** ([10, 9, 2, 5, 3, 7, 101, 18]):
```
Index:  0   1  2  3  4  5   6   7
nums:  10   9  2  5  3  7  101  18
dp:     1   1  1  2  2  3   4   4
```
- dp[3]=2: [2,5]
- dp[5]=3: [2,5,7] or [2,3,7]
- dp[6]=4: [2,5,7,101] or [2,3,7,101]
- Answer: 4

---

### Approach 2: Binary Search O(n log n)

**Core Idea**: Maintain an array `tails` where `tails[i]` stores the smallest ending element of all increasing subsequences of length `i+1`.

**Key Insight**: If we have two increasing subsequences of the same length, we prefer the one with the smaller tail because it has more potential to be extended.

1. **Invariant**: `tails` is always sorted in increasing order

2. **For each number `num` in nums**:
   - Binary search in `tails` to find the leftmost position where `tails[pos] >= num`
   - If pos == len(tails): append num (extends the longest subsequence)
   - Else: replace `tails[pos] = num` (found a better tail for this length)

3. **Why this works**:
   - If num is larger than all tails: extends the longest subsequence
   - If num replaces tails[pos]: we now have a better (smaller) tail for length pos+1
   - The length of tails at the end is the LIS length

**Example Walkthrough** ([10, 9, 2, 5, 3, 7, 101, 18]):
```
num=10:  tails=[10]
num=9:   tails=[9]      (replace 10 with 9, better tail for length 1)
num=2:   tails=[2]      (replace 9 with 2)
num=5:   tails=[2,5]    (5 > 2, extend)
num=3:   tails=[2,3]    (replace 5 with 3, better tail for length 2)
num=7:   tails=[2,3,7]  (7 > 3, extend)
num=101: tails=[2,3,7,101] (extend)
num=18:  tails=[2,3,7,18]  (replace 101 with 18)
```
Length = 4

**Note**: The final `tails` array is NOT necessarily the actual LIS (e.g., [2,3,7,18] is valid, but so is [2,5,7,101]). It only guarantees the correct length.

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Brute force (recursion) | O(2^n) | O(n) | Check all subsequences |
| DP (bottom-up) | O(n^2) | O(n) | Standard DP approach |
| Binary search + DP | O(n log n) | O(n) | Optimal for length only |
| Patience sorting | O(n log n) | O(n) | Can reconstruct actual LIS |

In [ ]:
from typing import List
import bisect

class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        """
        Binary Search approach - O(n log n)
        
        Time: O(n log n)
        Space: O(n)
        """
        tails = []
        
        for num in nums:
            # Find leftmost position where tails[pos] >= num
            pos = bisect.bisect_left(tails, num)
            
            if pos == len(tails):
                # num is larger than all elements in tails, extend
                tails.append(num)
            else:
                # Replace with smaller tail for this length
                tails[pos] = num
        
        return len(tails)
    
    def lengthOfLIS_DP(self, nums: List[int]) -> int:
        """
        Dynamic Programming approach - O(n^2)
        
        Time: O(n^2)
        Space: O(n)
        """
        if not nums:
            return 0
        
        n = len(nums)
        dp = [1] * n  # Each element forms a subsequence of length 1
        
        for i in range(1, n):
            for j in range(i):
                if nums[j] < nums[i]:
                    dp[i] = max(dp[i], dp[j] + 1)
        
        return max(dp)

In [ ]:
# Test cases
tests = [
    ([10,9,2,5,3,7,101,18], 4),    # [2,3,7,101] or [2,5,7,101]
    ([0,1,0,3,2,3], 4),            # [0,1,2,3]
    ([7,7,7,7,7,7,7], 1),          # Only one element (strictly increasing)
    ([1,3,6,7,9,4,10,5,6], 6),     # [1,3,6,7,9,10]
    ([4,10,4,3,8,9], 3),           # [4,8,9] or [3,8,9]
    ([1], 1),                      # Single element
    ([2,1], 1),                    # Decreasing
]

solver = Solution()
for nums, expected in tests:
    result_bs = solver.lengthOfLIS(nums)
    result_dp = solver.lengthOfLIS_DP(nums)
    assert result_bs == expected, f"Binary Search failed for {nums}: got {result_bs}, expected {expected}"
    assert result_dp == expected, f"DP failed for {nums}: got {result_dp}, expected {expected}"
print('All tests passed for both approaches.')

## Complexity Analysis

**Binary Search Approach:**
- **Time**: O(n log n) - n iterations, each with O(log n) binary search
- **Space**: O(n) - tails array in worst case has n elements (fully increasing sequence)

**DP Approach:**
- **Time**: O(n^2) - nested loops over the array
- **Space**: O(n) - dp array stores state for each position

## Edge Cases & Pitfalls
- **All equal elements**: LIS length is 1 (strictly increasing means no duplicates)
- **Strictly vs non-strictly increasing**: This problem requires strict inequality; adjust if problem allows equals
- **Empty array**: Return 0
- **Single element**: Return 1
- **Descending order**: LIS is 1 (any single element)
- **Reconstructing the actual LIS**: Binary search approach gives length only; need additional bookkeeping to reconstruct the sequence
- **bisect_left vs bisect_right**: Use bisect_left for strict inequality to find the leftmost insertion point

## Follow-up Variants
- **Reconstruct the actual LIS**: Modify binary search to track predecessors, or use patience sorting
- **Number of LIS**: Count all distinct LIS (LC 673)
- **Longest non-decreasing subsequence**: Allow equal elements
- **2D LIS**: Given pairs, find longest chain where both dimensions increase (LC 354 - Russian Doll Envelopes)
- **Circular array**: LIS in a circular array where you can start from any position
- **Longest decreasing subsequence**: Reverse the inequality
- **Longest bitonic subsequence**: First increasing then decreasing

## Takeaways
- LIS is one of the most fundamental DP problems and appears frequently in interviews
- The O(n^2) DP solution is intuitive and demonstrates clear state transitions
- The O(n log n) binary search optimization is a beautiful example of using auxiliary data structures to improve complexity
- Understanding the "tails" array invariant is key: it maintains the smallest tail for each possible length
- The binary search approach shows how to transform a DP problem into a search problem
- This pattern of maintaining optimal candidates (smallest/largest) at each step appears in many optimization problems
- Mastering both approaches demonstrates deep understanding of DP and optimization techniques

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 673 | Number of Longest Increasing Subsequence | DP with count tracking |
| LC 354 | Russian Doll Envelopes | 2D LIS with sorting |
| LC 646 | Maximum Length of Pair Chain | Similar greedy/DP |
| LC 1143 | Longest Common Subsequence | Related subsequence problem |
| LC 674 | Longest Continuous Increasing Subsequence | Easier variant (contiguous) |